In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import recall_score, precision_score, average_precision_score, roc_auc_score, confusion_matrix

In [3]:
df = pd.read_csv("credit_card_fraud_10k.csv")

print(df.shape)
print(df.head())
print(df.dtypes)
print(df["is_fraud"].value_counts())

(10000, 10)
   transaction_id  amount  transaction_hour merchant_category  \
0               1   84.47                22       Electronics   
1               2  541.82                 3            Travel   
2               3  237.01                17           Grocery   
3               4  164.33                 4           Grocery   
4               5   30.53                15              Food   

   foreign_transaction  location_mismatch  device_trust_score  \
0                    0                  0                  66   
1                    1                  0                  87   
2                    0                  0                  49   
3                    0                  1                  72   
4                    0                  0                  79   

   velocity_last_24h  cardholder_age  is_fraud  
0                  3              40         0  
1                  1              64         0  
2                  1              61         0  
3         

In [ ]:
print("Parte 2: calidad de datos")
print("Nulos totales:", df.isna().sum().sum())
print("Duplicados:", df.duplicated().sum())

for c in ["amount", "velocity_last_24h"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    out = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).mean()
    print(f"Outliers {c}: {out:.2%}")

Parte 2: calidad de datos
Nulos totales: 0
Duplicados: 0
Outliers amount: 5.01%
Outliers velocity_last_24h: 0.51%


In [11]:
print("Parte 3: tasa de fraude por variable binaria")
for c in ["foreign_transaction", "location_mismatch"]:
    print(pd.crosstab(df[c], df["is_fraud"], normalize="index").round(4))

Parte 3: tasa de fraude por variable binaria
is_fraud                  0       1
foreign_transaction                
0                    0.9924  0.0076
1                    0.9162  0.0838
is_fraud                0       1
location_mismatch                
0                  0.9914  0.0086
1                  0.9160  0.0840


In [13]:
print("Parte 4: separar y entrenar")
TARGET = "is_fraud"
X = df.drop(columns=["transaction_id", TARGET])
y = df[TARGET]
CAT = ["merchant_category"]
num_feats = [c for c in X.columns if c not in CAT]
SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)

def preprocesar():
    return ColumnTransformer([
        ("num", StandardScaler(), num_feats),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])

modelos = {
    "Regresión Logística": make_pipeline(preprocesar(),
        LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": make_pipeline(preprocesar(),
        RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                               random_state=SEED, n_jobs=-1)),
    "SVM (RBF)": make_pipeline(preprocesar(),
        SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"recall": "recall", "precision": "precision",
           "pr_auc": "average_precision", "roc_auc": "roc_auc"}

for nombre, modelo in modelos.items():
    res = cross_validate(modelo, X_train, y_train, cv=cv, scoring=scoring)
    print(nombre, {k: f"{res['test_'+k].mean():.3f} ± {res['test_'+k].std():.3f}" for k in scoring})

Parte 4: separar y entrenar
Train: (8000, 8) Test: (2000, 8)
Regresión Logística {'recall': '0.967 ± 0.017', 'precision': '0.273 ± 0.017', 'pr_auc': '0.665 ± 0.095', 'roc_auc': '0.992 ± 0.003'}
Random Forest {'recall': '0.421 ± 0.120', 'precision': '1.000 ± 0.000', 'pr_auc': '0.966 ± 0.029', 'roc_auc': '0.999 ± 0.001'}
SVM (RBF) {'recall': '0.859 ± 0.068', 'precision': '0.569 ± 0.040', 'pr_auc': '0.820 ± 0.093', 'roc_auc': '0.996 ± 0.002'}


In [14]:
print("Parte 5: test y lectura de negocio")
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    print(nombre, "recall:", round(recall_score(y_test, pred), 3),
          "precision:", round(precision_score(y_test, pred, zero_division=0), 3))

regla = ((X_test["foreign_transaction"] == 1) | (X_test["location_mismatch"] == 1)).astype(int)
print("Regla simple recall:", round(recall_score(y_test, regla), 3),
      "precision:", round(precision_score(y_test, regla, zero_division=0), 3))

pred_svm = modelos["SVM (RBF)"].predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, pred_svm).ravel()
alertas = tp + fp
print(f"Alertas: {alertas} | fraudes en alertas: {tp} | de cada 100 alertas: {100*tp/alertas:.1f} son fraude")
print(f"Fraudes detectados: {tp} de {tp+fn} ({100*tp/(tp+fn):.1f}%) | falsas alarmas: {fp}")

Parte 5: test y lectura de negocio
Regresión Logística recall: 0.967 precision: 0.259
Random Forest recall: 0.533 precision: 1.0
SVM (RBF) recall: 0.833 precision: 0.521
Regla simple recall: 0.867 precision: 0.076
Alertas: 48 | fraudes en alertas: 25 | de cada 100 alertas: 52.1 son fraude
Fraudes detectados: 25 de 30 (83.3%) | falsas alarmas: 23
